## Module 01: Data Ingestion, Integrity Audit & Preprocessing Pipeline
## Project: Washington State Electric Vehicle Population Analysis


---

### 1. Module Overview
This notebook establishes the foundational data engineering and cleaning pipeline for the Washington State Electric Vehicle (EV) Population dataset. Real-world administrative records frequently contain structural anomalies, high-cardinality metadata identifiers, and missing entries. The objective of this module is to ingest raw registry records, audit schema validity, clean categorical features, eliminate zero-variance administrative features, and export an analytically rigorous dataset.

### 2. Methodological Pipeline:
1. **Raw Ingestion & Schema Inspection:** Load CSV records, examine dimensional matrices, and evaluate memory utilization.
2. **Feature Name Normalization:** Standardize arbitrary headers into uniform lowercase snake_case formatting.
3. **Missing Value Diagnostics:** Conduct exhaustive sparsity audits to identify null frequency and implement deterministic dropping/imputation heuristics.
4. **Dimensionality Reduction:** Strip administrative tracking attributes (VIN prefixes, registration IDs, census tracts) that offer zero explanatory variance.
5. **Categorical & Temporal Standardization:** Discretize powertrain mechanisms into binary cohorts (BEV vs. PHEV) and filter historical focus to the modern adoption era (2011–present).
6. **Artifact Export:** Save the normalized dataset (`ev_cleaned.csv`) for exploratory analysis and downstream modeling.

### load labries & ingest dataset

In [9]:
import numpy as np
import pandas as pd

# load raw dataset from the data folder.

df = pd.read_csv("../data/Electric_Vehicle_Population_Data.csv")

print("dataset shape:", df.shape)
print("dataset data type:", df.dtypes)
df.head()
df.tail()

dataset shape: (150482, 17)
dataset data type: VIN (1-10)                                            object
County                                                object
City                                                  object
State                                                 object
Postal Code                                          float64
Model Year                                             int64
Make                                                  object
Model                                                 object
Electric Vehicle Type                                 object
Clean Alternative Fuel Vehicle (CAFV) Eligibility     object
Electric Range                                         int64
Base MSRP                                              int64
Legislative District                                 float64
DOL Vehicle ID                                         int64
Vehicle Location                                      object
Electric Utility                      

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Base MSRP,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
150477,WBY43AW05P,Grays Harbor,Montesano,WA,98563.0,2023,BMW,I4,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,19.0,251204075,POINT (-123.60535 46.982215),BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF G...,5.302700e+10
150478,5YJ3E1EB7P,King,Seattle,WA,98104.0,2023,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,43.0,241344414,POINT (-122.329075 47.6018),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
150479,5YJYGDEEXM,King,Seattle,WA,98109.0,2021,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,43.0,180705626,POINT (-122.34848 47.632405),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
150480,5UXTA6C08P,Snohomish,Mountlake Terrace,WA,98043.0,2023,BMW,X5,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,30,0,1.0,240473950,POINT (-122.30842 47.78416),PUGET SOUND ENERGY INC,5.306105e+10
150481,7SAYGDEF8N,Skagit,Mount Vernon,WA,98273.0,2022,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0,0,40.0,207667589,POINT (-122.338975 48.41333),PUGET SOUND ENERGY INC,5.305795e+10


In [10]:
# convert the all columns to the lowercase and replace with underscore
df.columns = df.columns.str.lower().str.replace(" " ,"_").str.replace("-","_")

print("cleaned columns:")
print(list(df.columns))

cleaned columns:
['vin_(1_10)', 'county', 'city', 'state', 'postal_code', 'model_year', 'make', 'model', 'electric_vehicle_type', 'clean_alternative_fuel_vehicle_(cafv)_eligibility', 'electric_range', 'base_msrp', 'legislative_district', 'dol_vehicle_id', 'vehicle_location', 'electric_utility', '2020_census_tract']


In [11]:
# find the columns with the missing value and display the missing count and percentage.

missing_data = df.isnull().sum()

missing_df  = pd.DataFrame({
    "missing data": missing_data,
    "percentage": (missing_data/ len(df))*100
})


missing_df

,missing data,percentage
vin_(1_10),0,0.000000
county,3,0.001994
city,3,0.001994
state,0,0.000000
postal_code,3,0.001994
model_year,0,0.000000
make,0,0.000000
model,0,0.000000
electric_vehicle_type,0,0.000000
clean_alternative_fuel_vehicle_(cafv)_eligibility,0,0.000000


In [12]:
df_clean = df.drop(columns=["vin_(1_10)", "dol_vehicle_id", 'vehicle_location', '2020_census_tract' ,"legislative_district"])
df_clean.shape
df_clean.head()


,county,city,state,postal_code,model_year,make,model,electric_vehicle_type,clean_alternative_fuel_vehicle_(cafv)_eligibility,electric_range,base_msrp,electric_utility
0,King,Seattle,WA,98103.0,2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,258,0,CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA)
1,King,Bothell,WA,98011.0,2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)
2,Yakima,Yakima,WA,98908.0,2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,PACIFICORP
3,King,Kirkland,WA,98034.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)
4,Thurston,Olympia,WA,98501.0,2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,97,0,PUGET SOUND ENERGY INC


### data filtering

In [13]:
# Drop rows where critical fields like Make, Model, or County are missing
df_clean = df_clean.dropna(subset=['make', 'model', 'model_year', 'county'])

# Keep modern EVs from model year 2011 onwards
df_clean = df_clean[df_clean['model_year'] >= 2011]


### domain feature Engneering

In [14]:

# 1. mapping powertrain type
mapping = {
    'Battery Electric Vehicle (BEV)':"BEV",
    "Plug-in Hybrid Electric Vehicle (PHEV)":"PHEV"
}

df_clean["powertrain"] = df_clean["electric_vehicle_type"].map(mapping)

# 2. Market Adoption Era
def get_era(year):
    if year <= 2016:
        return 'Early Adoption (2011-2016)'
    elif year <= 2020:
        return 'Transition Phase (2017-2020)'
    else:
        return 'Mass Market (2021-Present)'

df_clean['adoption_era'] = df_clean['model_year'].apply(get_era)

# 3. Discretized Range Categorization (for models with certified ratings)
def get_range_bracket(r):
    if r == 0:
        return 'Unrecorded / Not Certified'
    elif r < 100:
        return 'Short Range (<100 mi)'
    elif r <= 200:
        return 'Mid Range (100-200 mi)'
    else:
        return 'Long Range (>200 mi)'

df_clean['range_category'] = df_clean['electric_range'].apply(get_range_bracket)

print("Engineered Feature Value Distributions:")
print("\n--- Adoption Eras ---")
print(df_clean['adoption_era'].value_counts())
print("\n--- Powertrain Distribution ---")
print("\n--- Range Brackets ---")
print(df_clean['range_category'].value_counts())

Engineered Feature Value Distributions:

--- Adoption Eras ---
adoption_era
Mass Market (2021-Present)      84204
Transition Phase (2017-2020)    45025
Early Adoption (2011-2016)      21191
Name: count, dtype: int64

--- Powertrain Distribution ---

--- Range Brackets ---
range_category
Unrecorded / Not Certified    69698
Short Range (<100 mi)         43446
Long Range (>200 mi)          30628
Mid Range (100-200 mi)         6648
Name: count, dtype: int64


In [15]:
# save cleaned data

df_clean.to_csv("../data/ev_cleaned_dataset.csv",index=False)

print("dataset save successfully!")

dataset save successfully!
